# 32 — Random Projection to WJ Simplex 512

**Baseline:** Random Gaussian projection (18k → 512), shifted L1-simplex normalization, HNSW + WeightedJaccard.

No learning — shows the gap between random and learned compression. If PCA (0.43 R@10) and random projection are both low, it confirms the MLP's gains come from WJ-aware training, not just dimensionality reduction.

In [1]:
import sys, time
import numpy as np
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup, eval_recall, shifted_l1_simplex, load_dataset,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name  = "full"
out_dim       = 512
THREADS       = 150
seed          = 42
candidate_ks  = [500, 1000] if dataset_name == "10k" else [1000, 2000]
METHOD_NAME   = "random_projection_wj_512"
NOTEBOOK_NAME = "31_random_projection_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_random_proj_wj_512.pkl"

np.random.seed(seed)
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)


dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)


In [2]:
# Random Gaussian projection: 18k → 512, scaled by 1/sqrt(out_dim)
rng = np.random.default_rng(seed)
W   = rng.standard_normal((qt.shape[1], out_dim)).astype(np.float32) / np.sqrt(out_dim)

print(f"Projecting {qt.shape} -> (N, {out_dim}) ...", flush=True)
proj = qt.astype(np.float32) @ W           # (N, 512), has negatives
embs = shifted_l1_simplex(proj)            # shift to nonneg, L1-normalize
print(f"embs={embs.shape} | sum check: min={embs.sum(1).min():.4f} max={embs.sum(1).max():.4f}")


Projecting (233773, 18220) -> (N, 512) ...
embs=(233773, 512) | sum check: min=0.0088 max=1.0000


In [3]:
corpus_embs = embs[:query_start]
query_embs  = embs[query_start:]
max_k = max(max(candidate_ks), 500)

nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
print("\n--- Stage 1: WJ HNSW on 512-D random projection ---")
for k, v in metrics.items():
    if isinstance(k, int): print(f"  R@{k:<4} = {v:.4f}")
print(f"  QPS={metrics['qps']:.1f}")
save_result(OUT_PATH, dataset_name, METHOD_NAME, metrics, meta={"notebook": NOTEBOOK_NAME})

preload_rerank_corpus(corpus_qt, corpus_sums)
for ck in candidate_ks:
    cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
    t0 = time.time()
    rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
    qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
    rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
    key = f"{METHOD_NAME}_rerank_{ck}"
    print(f"\n--- Stage 2: top-{ck} candidates + raw WJ rerank ---")
    for k, v in rr_metrics.items():
        if isinstance(k, int): print(f"  R@{k:<4} = {v:.4f}")
    save_result(OUT_PATH, dataset_name, key, rr_metrics, meta={"notebook": NOTEBOOK_NAME})
release_rerank_corpus()

cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*


--- Stage 1: WJ HNSW on 512-D random projection ---
  R@10   = 0.6281
  R@50   = 0.6632
  R@100  = 0.6776
  R@500  = 0.7218
  QPS=1259.5
saved random_projection_wj_512 -> /tmp/results_sota_random_proj_wj_512.pkl
Corpus pre-loaded to GPU: 12.69 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*****************************************************




--- Stage 2: top-1000 candidates + raw WJ rerank ---
  R@10   = 0.9893
  R@50   = 0.9849
  R@100  = 0.9757
  R@500  = 0.8718
saved random_projection_wj_512_rerank_1000 -> /tmp/results_sota_random_proj_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************


--- Stage 2: top-2000 candidates + raw WJ rerank ---
  R@10   = 0.9911
  R@50   = 0.9913
  R@100  = 0.9881
  R@500  = 0.9273
saved random_projection_wj_512_rerank_2000 -> /tmp/results_sota_random_proj_wj_512.pkl
